In [1]:
!pip -q install -U langchain langgraph langchain-openai pydantic ipython

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.9/107.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.9/471.9 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 54.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.7/625.7 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.7/508.7 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.12.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.0 which is inco

In [2]:
import os
import json
from getpass import getpass
from typing import TypedDict, List, Dict, Any, Literal

from IPython.display import display, Markdown
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

### Define structured schemas

In [3]:
class UserInput(BaseModel):
    user_goal: str
    recipient_name: str = ""
    recipient_role: str = ""
    sender_name: str = ""
    sender_role: str = ""
    relationship_context: str = ""
    message_context: str = ""
    key_points: List[str] = Field(default_factory=list)
    tone_preference: str = ""
    desired_length: Literal["short", "medium", "long"] = "medium"
    include_cta: bool = False
    edit_requests: List[str] = Field(default_factory=list)

class TaskClassification(BaseModel):
    intent: Literal[
        "job_application",
        "client_communication",
        "support_reply",
        "meeting_followup",
        "general_email"
    ]
    recommended_tone: str
    subject_needed: bool
    reply_mode: bool
    reason: str

In [4]:
class MissingDetails(BaseModel):
    missing_fields: List[str]
    placeholders: List[str]
    assumptions: List[str]

class TemplatePlan(BaseModel):
    template_name: str
    subject_style: str
    opening_style: str
    body_structure: List[str]
    closing_style: str

class DraftOutput(BaseModel):
    subject: str
    draft: str

class RefinedOutput(BaseModel):
    improved_version: str
    short_version: str
    formal_version: str
    cta_line: str

class FinalPackage(BaseModel):
    intent: str
    tone: str
    subject: str
    missing_fields: List[str]
    placeholders: List[str]
    assumptions: List[str]
    initial_draft: str
    improved_version: str
    short_version: str
    formal_version: str
    cta_line: str

### Create helper functions

In [5]:

def clean_text(value):
    return str(value).strip()

def clean_list(value):
    if isinstance(value, list):
        return [str(item).strip() for item in value if str(item).strip()]
    if isinstance(value, str):
        parts = [item.strip() for item in value.split("\n")]
        parts = [item for item in parts if item]
        if parts:
            return parts
    return []

def normalize_input_data(data):
    payload = UserInput(
        user_goal=clean_text(data.get("user_goal", "")),
        recipient_name=clean_text(data.get("recipient_name", "")),
        recipient_role=clean_text(data.get("recipient_role", "")),
        sender_name=clean_text(data.get("sender_name", "")),
        sender_role=clean_text(data.get("sender_role", "")),
        relationship_context=clean_text(data.get("relationship_context", "")),
        message_context=clean_text(data.get("message_context", "")),
        key_points=clean_list(data.get("key_points", [])),
        tone_preference=clean_text(data.get("tone_preference", "")),
        desired_length=clean_text(data.get("desired_length", "medium")) or "medium",
        include_cta=bool(data.get("include_cta", False)),
        edit_requests=clean_list(data.get("edit_requests", []))
    )
    return payload.model_dump()

In [6]:
def make_template(template_name, subject_style, opening_style, body_structure, closing_style):
    plan = TemplatePlan(
        template_name=template_name,
        subject_style=subject_style,
        opening_style=opening_style,
        body_structure=body_structure,
        closing_style=closing_style
    )
    return plan.model_dump()

In [7]:
def build_markdown_output(data):
    lines = []
    lines.append("# Professional Email Draft Agent Output")
    lines.append("")
    lines.append("## Detected Intent")
    lines.append(data["intent"])
    lines.append("")
    lines.append("## Recommended Tone")
    lines.append(data["tone"])
    lines.append("")
    lines.append("## Missing Details")
    if data["missing_fields"]:
        for item in data["missing_fields"]:
            lines.append(f"- {item}")
    else:
        lines.append("- No major missing details detected")
    lines.append("")
    lines.append("## Suggested Placeholders")
    if data["placeholders"]:
        for item in data["placeholders"]:
            lines.append(f"- {item}")
    else:
        lines.append("- No placeholder needed")
    lines.append("")
    lines.append("## Assumptions Used")
    if data["assumptions"]:
        for item in data["assumptions"]:
            lines.append(f"- {item}")
    else:
        lines.append("- No strong assumptions used")
    lines.append("")
    lines.append("## Subject")
    lines.append(data["subject"])
    lines.append("")
    lines.append("## Initial Draft")
    lines.append(data["initial_draft"])
    lines.append("")
    lines.append("## Improved Version")
    lines.append(data["improved_version"])
    lines.append("")
    lines.append("## Short Version")
    lines.append(data["short_version"])
    lines.append("")
    lines.append("## Formal Version")
    lines.append(data["formal_version"])
    lines.append("")
    lines.append("## CTA Line")
    lines.append(data["cta_line"] if data["cta_line"] else "No CTA line added")
    return "\n".join(lines)

### Define workflow state

In [8]:
class EmailState(TypedDict):
    raw_input: Dict[str, Any]
    input_data: Dict[str, Any]
    classification: Dict[str, Any]
    missing: Dict[str, Any]
    template: Dict[str, Any]
    initial_draft: Dict[str, Any]
    refined: Dict[str, Any]
    final_output: Dict[str, Any]
    report_markdown: str

### Create graph nodes

In [10]:
def normalize_input_node(state: EmailState):
    data = normalize_input_data(state["raw_input"])
    return {
        "input_data": data
    }

def classify_task_node(state: EmailState):
    model = llm.with_structured_output(TaskClassification)

    prompt = f"""
You classify professional email drafting tasks.

Input data:
{json.dumps(state["input_data"], indent=2, ensure_ascii=False)}

Decide:
- the primary intent
- the best professional tone
- whether a subject is needed
- whether this is mainly a reply
- a short reason

Choose one intent from:
job_application
client_communication
support_reply
meeting_followup
general_email
"""
    result = model.invoke(prompt)

    return {
        "classification": result.model_dump()
    }

def detect_missing_details_node(state: EmailState):
    model = llm.with_structured_output(MissingDetails)

    prompt = f"""
You review a professional email drafting request.

Input data:
{json.dumps(state["input_data"], indent=2, ensure_ascii=False)}

Task classification:
{json.dumps(state["classification"], indent=2, ensure_ascii=False)}

Identify useful missing details.
Do not ask questions.
Instead:
- list missing fields
- suggest placeholders that can safely appear in the draft
- list practical assumptions
"""
    result = model.invoke(prompt)

    return {
        "missing": result.model_dump()
    }

def route_template(state: EmailState):
    intent = state["classification"]["intent"]

    if intent == "job_application":
        return "job_application_template"
    if intent == "client_communication":
        return "client_template"
    if intent == "support_reply":
        return "support_template"
    if intent == "meeting_followup":
        return "followup_template"
    return "general_template"

def job_application_template_node(state: EmailState):
    return {
        "template": make_template(
            template_name="job_application",
            subject_style="clear role-focused application subject",
            opening_style="respectful and confident introduction",
            body_structure=[
                "why you are writing",
                "who you are",
                "most relevant skills or projects",
                "why you fit the opportunity",
                "call to action"
            ],
            closing_style="professional and motivated close"
        )
    }

def client_template_node(state: EmailState):
    return {
        "template": make_template(
            template_name="client_communication",
            subject_style="business subject with purpose",
            opening_style="polite client-friendly opener",
            body_structure=[
                "context",
                "main update or request",
                "value or benefit",
                "next step",
                "call to action"
            ],
            closing_style="clear and professional business close"
        )
    }

def support_template_node(state: EmailState):
    return {
        "template": make_template(
            template_name="support_reply",
            subject_style="issue-oriented subject if needed",
            opening_style="empathetic acknowledgment",
            body_structure=[
                "acknowledge the issue",
                "clarify what is known",
                "state the solution or next step",
                "timeline or expectation",
                "offer further help"
            ],
            closing_style="supportive and reassuring close"
        )
    }

def followup_template_node(state: EmailState):
    return {
        "template": make_template(
            template_name="meeting_followup",
            subject_style="meeting follow-up subject",
            opening_style="warm and appreciative opener",
            body_structure=[
                "thank them for the meeting",
                "brief recap of discussion",
                "next steps",
                "required action if any",
                "call to action"
            ],
            closing_style="professional and collaborative close"
        )
    }

def general_template_node(state: EmailState):
    return {
        "template": make_template(
            template_name="general_email",
            subject_style="simple purpose-driven subject",
            opening_style="polite opener",
            body_structure=[
                "reason for writing",
                "key details",
                "request or update",
                "next step if needed"
            ],
            closing_style="clean professional close"
        )
    }

def generate_draft_node(state: EmailState):
    model = llm.with_structured_output(DraftOutput)

    prompt = f"""
You are drafting a professional email.

Input data:
{json.dumps(state["input_data"], indent=2, ensure_ascii=False)}

Task classification:
{json.dumps(state["classification"], indent=2, ensure_ascii=False)}

Missing details:
{json.dumps(state["missing"], indent=2, ensure_ascii=False)}

Template:
{json.dumps(state["template"], indent=2, ensure_ascii=False)}

Rules:
- Write a polished email draft
- Keep the tone aligned with recommended_tone
- Respect tone_preference when it helps
- Respect desired_length
- Use placeholders like [Recipient Name] only when needed
- Use the key points naturally
- Add a CTA near the end when include_cta is true
- Respect edit_requests
- Return subject and draft body only
"""
    result = model.invoke(prompt)

    return {
        "initial_draft": result.model_dump()
    }

def refine_draft_node(state: EmailState):
    model = llm.with_structured_output(RefinedOutput)

    prompt = f"""
You are refining a professional email.

Input data:
{json.dumps(state["input_data"], indent=2, ensure_ascii=False)}

Task classification:
{json.dumps(state["classification"], indent=2, ensure_ascii=False)}

Initial draft:
{json.dumps(state["initial_draft"], indent=2, ensure_ascii=False)}

Create:
- improved_version: best practical version
- short_version: shorter but still complete
- formal_version: more formal and polished
- cta_line: one standalone CTA line that can be inserted near the end

Rules:
- Keep the message aligned with the original goal
- Do not remove important details without reason
- Respect edit_requests such as make more professional, make shorter, add CTA
- Keep placeholders if key details are missing
"""
    result = model.invoke(prompt)

    return {
        "refined": result.model_dump()
    }

def assemble_output_node(state: EmailState):
    package = FinalPackage(
        intent=state["classification"]["intent"],
        tone=state["classification"]["recommended_tone"],
        subject=state["initial_draft"]["subject"],
        missing_fields=state["missing"]["missing_fields"],
        placeholders=state["missing"]["placeholders"],
        assumptions=state["missing"]["assumptions"],
        initial_draft=state["initial_draft"]["draft"],
        improved_version=state["refined"]["improved_version"],
        short_version=state["refined"]["short_version"],
        formal_version=state["refined"]["formal_version"],
        cta_line=state["refined"]["cta_line"]
    )

    data = package.model_dump()
    markdown = build_markdown_output(data)

    return {
        "final_output": data,
        "report_markdown": markdown
    }